# Comparative SICR Protest Modeling\nMinimal implementation with progress tracking.

In [ ]:

import numpy as np
import pandas as pd
from scipy.integrate import solve_ivp
from scipy.optimize import differential_evolution
import matplotlib.pyplot as plt
import seaborn as sns
from tqdm.notebook import tqdm
import warnings
warnings.filterwarnings('ignore')


In [ ]:

# Load Data
df_fr = pd.read_csv('../data/france_2023_weekly.csv')
df_fr['date'] = pd.to_datetime(df_fr['date'])
t_fr = (df_fr['date'] - df_fr['date'].min()).dt.days.values
N_fr = df_fr['mid'].values

df_ir = pd.read_csv('../data/iran_2022_weekly.csv')
df_ir['Date'] = pd.to_datetime(df_ir['Date'])
t_ir = (df_ir['Date'] - df_ir['Date'].min()).dt.days.values
daily_arrests = np.maximum(0, np.diff(df_ir['Number of Individuals Arrested'].values, prepend=0))
N_ir = daily_arrests * 50.0  # Proxy scaling


In [ ]:

# Model and Simulation
def make_P_func(protest_days):
    def P(t):
        val = 0.0
        for pd in protest_days:
            val += 0.5 * (np.tanh(10 * (t - (pd - 0.5))) - np.tanh(10 * (t - (pd + 0.5))))
        return min(1.0, val)
    return P

def sicr_rhs(t, y, params, P_func):
    S, I, C, R = y
    b1, b2, chi, d11, d21, gamma, C0, n, eps12, eps22 = params
    Pt = P_func(t)
    omega = (d21 + eps22 * Pt) / (1.0 + ((I + C) / C0)**n)
    inflow = S * (b1 * I + b2 * C)
    dS = -inflow + gamma * R
    dI = inflow - chi * I - (d11 + eps12 * Pt) * I
    dC = chi * I - C * omega
    dR = (d11 + eps12 * Pt) * I + C * omega - gamma * R
    return [dS, dI, dC, dR]

def simulate_sicr(y0, params, t_span, protest_days, t_eval=None):
    P_func = make_P_func(protest_days)
    return solve_ivp(
        sicr_rhs, t_span, y0, args=(params, P_func),
        method='LSODA', t_eval=t_eval, rtol=1e-3, atol=1e-6
    )


In [ ]:

# Fitting with Progress Bar
def loss(params, t_obs, N_obs, S0, protest_days):
    full_params = list(params)
    full_params.insert(5, 0.010) # fixed gamma
    full_params.insert(7, 4.0)   # fixed n
    y0 = [S0, max(1, N_obs[0]*0.1), max(1, N_obs[0]*0.9), 0]
    
    sol = simulate_sicr(y0, full_params, [0, t_obs[-1]+10], protest_days=protest_days, t_eval=t_obs)
    if not sol.success or len(sol.t) != len(t_obs): return 1e9
    
    N_sim = np.maximum(1, sol.y[1] + sol.y[2])
    N_obs_safe = np.maximum(1, N_obs)
    return np.mean((np.log(N_sim) - np.log(N_obs_safe))**2)

def fit_model(name, t_obs, N_obs, S0, protest_days):
    bounds = [
        (1e-7, 1e-4), (1e-9, 1e-5), (0.01, 0.5), (0.01, 0.5), (0.05, 0.5),
        (10000, S0*0.5), (0.0, 1.0), (0.0, 1.0)
    ]
    
    pbar = tqdm(total=15, desc=f"Fitting {name}")
    def cb(xk, convergence=None):
        pbar.update(1)
        
    result = differential_evolution(
        loss, bounds, args=(t_obs, N_obs, S0, protest_days),
        seed=42, popsize=15, maxiter=15, polish=True, workers=1, callback=cb
    )
    pbar.close()
    
    full_params = list(result.x)
    full_params.insert(5, 0.010)
    full_params.insert(7, 4.0)
    return full_params

print("Starting calibrations...")
params_fr = fit_model("France", t_fr, N_fr, 4000000, t_fr)
params_ir = fit_model("Iran", t_ir, N_ir, 6000000, t_ir)


In [ ]:

# Plotting
fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(15, 6))

y0_fr = [4000000, max(1, N_fr[0]*0.1), max(1, N_fr[0]*0.9), 0]
sol_fr = simulate_sicr(y0_fr, params_fr, [0, t_fr[-1]], protest_days=t_fr)
ax1.plot(t_fr, np.maximum(1, N_fr), 'o', label='Observed (France)')
ax1.plot(sol_fr.t, np.maximum(1, sol_fr.y[1] + sol_fr.y[2]), label='Fitted')
ax1.set_yscale('log')
ax1.set_title('France 2023 Protests')
ax1.legend()

y0_ir = [6000000, max(1, N_ir[0]*0.1), max(1, N_ir[0]*0.9), 0]
sol_ir = simulate_sicr(y0_ir, params_ir, [0, t_ir[-1]], protest_days=t_ir)
ax2.plot(t_ir, np.maximum(1, N_ir), 'o', label='Observed (Iran)')
ax2.plot(sol_ir.t, np.maximum(1, sol_ir.y[1] + sol_ir.y[2]), label='Fitted')
ax2.set_yscale('log')
ax2.set_title('Iran 2022 Protests')
ax2.legend()

plt.show()
